# Maybank Article NER Service — Colab Workflow
Ikuti urutan dari audit data sampai evaluasi. Jangan menganggap saran entitas otomatis sebagai gold label.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/Maybank_NER_Service')
%cd $PROJECT

In [ ]:
!pip install -r requirements.txt
!python -m spacy download en_core_web_sm

## 1. Audit dan pembersihan
Letakkan semua CSV asli di `data/raw/` sebelum menjalankan cell berikut.

In [ ]:
!python src/prepare_data.py

## 2. Buat template anotasi
Saran entitas dari baseline hanya titik awal. Periksa span, label, dan offset secara manual; tambahkan entitas yang terlewat; tandai `reviewed: true` hanya setelah diperiksa.

In [ ]:
!python src/make_annotation_template.py --n 100
print('Buka data/annotations/annotation_template.jsonl, tinjau, lalu simpan sebagai annotated.jsonl')

## 3. Validasi, split, training, evaluasi
Jalankan setelah `data/annotations/annotated.jsonl` sudah berisi anotasi yang ditinjau manusia.

In [ ]:
!python src/validate_annotations.py
!python src/split_data.py --seed 42 --train 0.70 --val 0.15 --test 0.15

In [ ]:
!python src/train_model.py --epochs 20

In [ ]:
!python src/evaluate_model.py --split validation
!python src/evaluate_model.py --split test

## 4. Artefak
Model tersimpan di `models/trained/`; laporan berada di `reports/`. Untuk demo web, unduh/sinkronkan folder proyek ke komputer, lalu ikuti README. Pastikan test set tidak digunakan untuk tuning setelah evaluasi final.